In [ ]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt

base_filename = 'initial_walk_test_10-08-2026_16-31-45_3.csv'
gt_filename = 'initial_walk_test_10-08-2026_16-31-45_3_ground_truth_ZVW.csv'         

data_path = '../data/measured_walks/'
gt_path = '../data/zero_velocity_windows/'
save_path = '../data/zero_velocity_windows/'

df_base = pd.read_csv(f'{data_path}{base_filename}')
df_ground_truth = pd.read_csv(f'{gt_path}{gt_filename}')

In [184]:
def detect_zvw(df, acc_thresh, gyro_thresh, var_thresh, var_window, min_dwell):
    acc_mag = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
    gyro_mag = np.sqrt(df['gx']**2 + df['gy']**2 + df['gz']**2)

    # near the threshold?
    acc_thresh_samples = np.abs(acc_mag - 1) <= acc_thresh
    # rotating?
    gyro_mag_samples = gyro_mag <= gyro_thresh

    # Rolling variance (.rolling <- creates a rolling window + .var() <- calculates the variance)
    # is it vibrating or sliding?
    # .fillna(1.0) - first samples of data dont have enough prev points to fill a window
    # so this add 1.0 values instead of the empty NaN values to fill the window.
    acc_var = acc_mag.rolling(window=var_window).var().fillna(1.0)
    var_cond = acc_var <= var_thresh

    is_zvw = acc_thresh_samples & gyro_mag_samples & var_cond

    # Create a unique ID for every consecutive block of True's or False's
    blocks = (is_zvw != is_zvw.shift()).cumsum()

    # Group by that ID, sum the True's, and check if it meets min_dwell
    final_zvw = is_zvw.groupby(blocks).transform('sum') >= min_dwell

    # True and the previous is False
    is_start = final_zvw & ~final_zvw.shift(1, fill_value=False)
    # True and the following if False
    is_end = final_zvw & ~final_zvw.shift(-1, fill_value=False)

    zvw_start_idxs = df['seq'][is_start].tolist()
    zvs_end_idxs = df['seq'][is_end].tolist()

    return acc_mag, gyro_mag, (zvw_start_idxs, zvs_end_idxs)

In [190]:
ACC_DEVIATION = 0.3    # Allowable deviation from 1.0g
GYRO_LIMIT = 75.0       # Maximum dps
VAR_LIMIT = 0.06        # Maximum rolling variance
VAR_WINDOW = 15         # Variance window to calculate
DWELL = 20               # Minimum consecutive samples

acc_mag, gyro_mag, predicted_zvws = detect_zvw(df_base, ACC_DEVIATION, GYRO_LIMIT, VAR_LIMIT, VAR_WINDOW, DWELL)

In [186]:
def export_boolean_mask_comparison(df_base, df_gt, predicted_zvws):
    pred_starts, pred_ends = predicted_zvws
    gt_starts, gt_ends = df_gt['window_start_seq'], df_gt['window_end_seq']

    # Create new empty False columns
    df_base['is_pred_zvw'] = False
    df_base['is_gt_zvw'] = False

    for start, end in zip(pred_starts, pred_ends):
        mask = (df_base['seq'] >= start) & (df_base['seq'] <= end)
        df_base.loc[mask, 'is_pred_zvw'] = True

    for start, end in zip(gt_starts, gt_ends):
        mask = (df_base['seq'] >= start) & (df_base['seq'] <= end)
        df_base.loc[mask, 'is_gt_zvw'] = True

    clean_filename = base_filename.replace('.csv', '_zvw_boolean_masks.csv')
    df_save_path = f'{save_path}{clean_filename}'
    
    df_base.to_csv(df_save_path, index=False)
    print(f"Saved boolean mask comparison to {df_save_path}")
    
    return df_base

export_boolean_mask_comparison(df_base, df_ground_truth, predicted_zvws)

Saved boolean mask comparison to ../../data/zero_velocity_windows/initial_walk_test_10-08-2026_16-31-45_3_zvw_boolean_masks.csv


,t_us,seq,ax,ay,az,gx,gy,gz,valid,t_imu_us,t_serial_us,missed,is_pred_zvw,is_gt_zvw
0,256757126,33167,-0.0312,-0.0186,1.0200,-0.9155,1.8921,-0.1221,1,476,874,472,False,False
1,256762118,33168,-0.0303,-0.0190,1.0195,-0.7935,1.9531,-0.1831,1,470,879,472,False,False
2,256767119,33169,-0.0308,-0.0195,1.0171,-0.7935,1.7700,-0.0610,1,470,863,472,False,False
3,256772124,33170,-0.0298,-0.0186,1.0176,-0.7935,1.6479,-0.1221,1,484,852,472,False,False
4,256777122,33171,-0.0312,-0.0166,1.0195,-0.6714,1.7700,-0.1831,1,466,873,472,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6381,288662126,39548,-0.0244,-0.0381,1.0220,0.2441,-0.7324,-0.1221,1,479,931,472,True,False
6382,288667125,39549,-0.0234,-0.0386,1.0229,0.0000,-0.6104,-0.1221,1,474,865,472,True,False
6383,288672119,39550,-0.0220,-0.0381,1.0225,0.0000,-0.4883,-0.1221,1,479,824,472,True,False
6384,288677127,39551,-0.0239,-0.0376,1.0200,-0.0610,-0.5493,-0.1221,1,483,805,472,True,False


In [ ]:
fig, ax1 = plt.subplots(figsize=(14,6))
ax1.plot(df_base['seq'], acc_mag, color="tab:blue", label='Acc Mag')
ax1.set_xlabel('Sequence')
ax1.set_ylabel('Acceleration Magnitude')

ax2 = ax1.twinx()
ax2.plot(df_base['seq'], gyro_mag, color='tab:red', label='Gyro Mag')
ax2.set_xlabel('Sequence')
ax2.set_ylabel('Gyroscope Magnitude')

starts, ends = predicted_zvws

# paint the zvws over the acc and gyro mags
for i, (start, end) in enumerate(zip(starts, ends)):
    if i == 0:
        ax1.axvspan(start, end, color='y', alpha=0.3, label='Predicted ZVWs')
    else:
        ax1.axvspan(start, end, color='y', alpha=0.3)

gt_starts, gt_ends = df_ground_truth['window_start_seq'], df_ground_truth['window_end_seq']
for i, (start, end) in enumerate(zip(gt_starts, gt_ends)):
    if i == 0:
        ax1.axvspan(start, end, color='black', alpha=0.2, label='Ground Truth')
    else:
        ax1.axvspan(start, end, color='black', alpha=0.3)
   
    
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

plt.title('Zero-Velocity Windows')
plt.tight_layout()

save_file = f'{save_path}{base_filename}'
plt.savefig(f"{save_file.replace('.csv', '_ZVW.png')}", dpi=120)

plt.show()

In [188]:
# Create a DataFrame to permanently link the sequences to their durations
df_durations = pd.DataFrame({
    'start_seq': starts,
    'end_seq': ends,
    'duration': np.array(ends) - np.array(starts)
})

# Because it's a DataFrame, the start_seq and end_seq are filtered automatically alongside the duration!
df_walking = df_durations[df_durations['duration'] < 200]

print(f"Number of unique windows: {len(df_walking)}")
print("=== Sequence Numbers Grouped by Duration ===")
# Group by duration, and combine the start sequences into a list
seqs_per_bin = df_walking.groupby('duration')['start_seq'].apply(list)

for duration, seq_list in seqs_per_bin.items():
    print(f"Duration {duration} samples ({len(seq_list)} windows):")
    print(f"  Starts at seqs: {seq_list}\n")

fig, ax3 = plt.subplots(figsize=(10, 6))

# Use the duration column from our filtered DataFrame
# We add + 1 to the max so the very last duration is included in the bins
bins = np.arange(df_walking['duration'].min(), df_walking['duration'].max() + 1)

ax3.hist(df_walking['duration'], bins=bins, color='tab:blue', edgecolor='black', label='Predicted ZVWs')
ax3.set_title('Distribution of Zero-Velocity Window Durations')
ax3.set_xlabel('Window Duration (in Sequence Samples)')
ax3.set_ylabel('Frequency (Number of Windows)')
ax3.legend(loc='upper right')

for container in ax3.containers:
    # Custom label list to hide '0' values
    labels = [str(int(bar.get_height())) if bar.get_height() > 0 else "" for bar in container]
    ax3.bar_label(container, labels=labels, padding=2, fontsize=9)

plt.tight_layout()

hist_save_path = f"{save_path}{base_filename.replace('.csv', '_duration_hist.png')}"
plt.savefig(hist_save_path, dpi=120)

plt.show()

Number of unique windows: 13
=== Sequence Numbers Grouped by Duration ===
Duration 71 samples (1 windows):
  Starts at seqs: [37619]

Duration 74 samples (1 windows):
  Starts at seqs: [37178]

Duration 75 samples (1 windows):
  Starts at seqs: [38059]

Duration 76 samples (4 windows):
  Starts at seqs: [37399, 37837, 38286, 38508]

Duration 77 samples (1 windows):
  Starts at seqs: [38728]

Duration 78 samples (1 windows):
  Starts at seqs: [36501]

Duration 79 samples (1 windows):
  Starts at seqs: [36725]

Duration 80 samples (1 windows):
  Starts at seqs: [36952]

Duration 86 samples (1 windows):
  Starts at seqs: [36268]

Duration 96 samples (1 windows):
  Starts at seqs: [38953]

